In [1]:
# Mahjong CONSTS
from mahjong.constants import EAST, SOUTH, WEST, NORTH, HAKU, HATSU, CHUN #former 4 are .WINDS, and all 7 are .HONOR_INDICES
from mahjong.constants import FIVE_RED_MAN, FIVE_RED_PIN, FIVE_RED_SOU # Red Dora number cards > .AKA_DORA_LIST
from mahjong.constants import DISPLAY_WINDS # Str display of the winds

# Mahjong methods/classes
from mahjong.hand_calculating.hand import HandCalculator
from mahjong.hand_calculating.hand_config import HandConfig, OptionalRules
from mahjong.meld import Meld
from mahjong.tile import TilesConverter

# imports
import random
import re
from typing import List, Optional, Tuple, Union
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import copy
import os
from pathlib import Path
import base64

In [2]:
TOTAL_CARD_NUM = 136 # 4 * [9pin + 9sozu + 9manzu + 3sanngennpai + 4jihai]
TILE_IMAGE_DIR = "./tile_imgs/"

In [3]:
# UTILITY FUNCTIONS

class MahjongConverter:
    """
    Utility class for Mahjong tile data conversions.
    All methods are static to serve as a functional engine for wrapper classes.
    """

    @staticmethod
    def to_34_array(tiles_136: List[int]) -> List[int]:
        tiles_34 = [0] * 34
        for t in tiles_136:
            tiles_34[t // 4] += 1
        return tiles_34

    @staticmethod
    def from_34_to_136(tiles_34: List[int]) -> List[int]:
        tiles_136 = []
        for tile_type, count in enumerate(tiles_34):
            base_id = tile_type * 4
            for i in range(min(count, 4)):
                tiles_136.append(base_id + i)
        return tiles_136

    @staticmethod
    def to_str(tiles_136: List[int]) -> str:
        if not tiles_136: return ""
        t34 = MahjongConverter.to_34_array(tiles_136)
        sections = [(0, 9, "m"), (9, 18, "p"), (18, 27, "s"), (27, 30, "z"), (30, 33, "d")]
        res = ""
        for start, end, suffix in sections:
            group = "".join(str(i - start + 1) * t34[i] for i in range(start, end) if t34[i] > 0)
            if group: res += f"{group}{suffix}"
        return res

    @staticmethod
    def to_136(hand_str: str) -> List[int]:
        t34 = [0] * 34
        suites = re.findall(r"(\d+)([mpsz])", hand_str)
        mapping = {"m": 0, "p": 9, "s": 18, "z": 27}
        for digits, suffix in suites:
            for d in digits:
                idx = (int(d) - 1) + mapping[suffix]
                if t34[idx] < 4: t34[idx] += 1
        return MahjongConverter.from_34_to_136(t34)

class MahjongBase:
    """Provides shared comparison and arithmetic logic."""
    def to_34(self) -> List[int]:
        raise NotImplementedError

    def to_ids(self) -> List[int]:
        raise NotImplementedError

    def __eq__(self, other: object) -> bool:
        if isinstance(other, (str, list, int)):
            if isinstance(other, str): other = MSPZD(other)
            elif isinstance(other, list): other = Hand136(other)
            elif isinstance(other, int): other = Hand136([other])
        
        if hasattr(other, 'to_34'):
            return self.to_34() == other.to_34()
        return False

    def __add__(self, other: Union['MahjongBase', str, list, int]) -> 'Hand136':
        """Allows combining hands using the + operator."""
        # Convert 'self' IDs
        self_ids = self.to_ids()
        
        # Convert 'other' to IDs
        if isinstance(other, MahjongBase):
            other_ids = other.to_ids()
        elif isinstance(other, str):
            other_ids = MahjongConverter.to_136(other)
        elif isinstance(other, list):
            other_ids = other
        elif isinstance(other, int):
            other_ids = [other]
        else:
            raise TypeError(f"Unsupported operand type for +: 'MahjongBase' and '{type(other).__name__}'")
            
        return Hand136(self_ids + other_ids)

    def __radd__(self, other: Union[str, list, int]) -> 'Hand136':
        """Handles cases where raw types are on the left side of +."""
        return self.__add__(other)

class Hand136(MahjongBase):
    def __init__(self, ids: Union[int, List[int]]):
        # Flatten list if needed and sort for consistency
        self.ids = sorted(ids) if isinstance(ids, list) else [ids]

    def to_34(self) -> List[int]:
        return MahjongConverter.to_34_array(self.ids)

    def to_ids(self) -> List[int]:
        return self.ids

    def to_mspz(self) -> 'MSPZD':
        return MSPZD(MahjongConverter.to_str(self.ids))
    
    def remove(self, id):
        self.ids.remove(id)
    
    def __repr__(self):
        return f"Hand136({self.ids})"
    
    def __len__(self) -> int:
        """Allows calling len(hand_obj)"""
        return len(self.ids)

    def __iter__(self):
        """Allows for tile in hand_obj: ..."""
        return iter(self.ids)
    
    def __getitem__(self, position):
        """
        Allows for calling via index
        """
        return self.ids[position]

class MSPZD(MahjongBase):
    def __init__(self, notation: str):
        self.notation = notation

    def to_34(self) -> List[int]:
        return MahjongConverter.to_34_array(MahjongConverter.to_136(self.notation))

    def to_ids(self) -> List[int]:
        return MahjongConverter.to_136(self.notation)

    def to_136(self) -> Hand136:
        return Hand136(self.to_ids())

    def __repr__(self):
        return f"MSPZD('{self.notation}')"

def make_random_hand() -> Hand136:
    """Random generation of tiles of size 13 and 1 drawn tile

    Returns
    -------
    List[Hand136]
        Hand136 len==13, Hand136 len==1
    """
    all_tiles_i = list(range(TOTAL_CARD_NUM))
    random.shuffle(all_tiles_i)
    hand_tiles = sorted(all_tiles_i[:13])
    drawn_tile = all_tiles_i[13]
    return Hand136(hand_tiles), Hand136(drawn_tile)


In [4]:
# GAME SIMULTION CLASS
class MahjongPlayer:
    def __init__(self, name: str, initial_score: int = 35000):
        self.name = name
        self.score = initial_score
        self.hand: Optional[Hand136] = None  # Current hidden tiles
        self.discards: List[Hand136] = []    # History of 136-IDs discarded
        self.melds: List[str] = []           # Calls like 'pon', 'chi', 'kan'
        self.is_riichi = False
        self.is_dealer = False
        self.wind: int #0=East, 1=South, 2=West, 3=North

    def decide_discard(self) -> Hand136:
        discard_tile = Hand136(random.choice(self.hand))
        return discard_tile

    def discard_tile(self, tile: Hand136):
        # Implementation to remove from self.hand and add to self.discards
        self.hand.remove(tile)
        self.discards.append(tile)
        return tile

    def __repr__(self):
        dealer_mark = " (Dealer)" if self.is_dealer else ""
        return f"[{self.name}]{dealer_mark} Score: {self.score}"
    
class MahjongTable:
    BA = ["East", "South", "West", "North"]
    def __init__(
            self,
            starting_score: int = 25000,
            ending_score: int = 30000,
            num_players: int = 4,
            player_names: list[str] = [],
            game_len: int = 2
            ):
        self.starting_score = starting_score
        self.ending_score = ending_score
        self.num_players = num_players
        self.player_names = player_names
        self.game_len = game_len # Tonpuu=1, Hanchan=2
        if len(player_names)  == 0:
            self.player_names = [f"P{i}" for i in range(self.num_players)]
        self.players = List[MahjongPlayer]
        self.wall: List[Hand136] = []
        self.dead_wall: List[int]
        self.turn_index = 0  # 0: East, 1: South, 2: West, 3: North
        self.round_number = 0
        # etc.
        self.dora_indicators: List[int]
        self.honba = 0 # Counter for dealer repeats, Renchan
        self.log = []
                
    @property
    def tiles_remaining(self) -> int:
        """The core metric for Game Theory: How many turns are left in the round?"""
        return len(self.wall)

    def is_game_over(self) -> bool:
        """Game ends when round_number reaches rule's game length"""
        return self.round_number == self.game_len

    def is_round_over(self) -> bool:
        """Hand ends when no tiles remain to be drawn."""
        return self.tiles_remaining == 0

    def setup_round(self):
        """Initializes a new hand: Shuffles wall, deals tiles, sets Dora."""
        # Reinitialize field
        self.wall = list([Hand136(n) for n in list(range(136))])
        random.shuffle(self.wall)
        self.dead_wall = [self.wall.pop() for _ in range(14)]
        self.dora_indicators = [self.dead_wall[0]] # First dora. Dora is self.dead_wall[:5]

        self.turn_index = self.round_number % self.num_players
        # round=0, turn_index=0, 0 1 2 3
        # round=1, turn_index=1, 3 0 1 2
        # round=2, turn_index=2, 2 3 0 1
        # round=3, turn_index=3, 1 2 3 0

        # Reinitialize all players
        self.players = [MahjongPlayer(name, self.starting_score) for name in self.player_names]
        dealer_idx = (self.round_number - 1) % 4
        for i, p in enumerate(self.players):
            p.hand = Hand136([self.wall.pop().ids[0] for _ in range(13)])
            p.is_dealer = (i == dealer_idx)
            p.wind = (i - dealer_idx - 1) % 4
            
        self._log_state("SETUP ROUND")

    def get_current_player(self) -> MahjongPlayer:
        return self.players[self.turn_index]

    def next_turn(self):
        self.turn_index = (self.turn_index + 1) % self.num_players

    def play_round(self):
        """A simplified loop representing one full hand (Kyoku)."""
        while not self.is_round_over():
            current_player = self.get_current_player()
            current_player: MahjongPlayer
            
            # DRAW
            new_tile = self.wall.pop()
            self._log_state("DRAW", new_tile.to_mspz())
            current_player.hand += new_tile

            # DISCARD (AI Decision Point)
            discarded_tile = current_player.decide_discard()
            current_player.discard_tile(discarded_tile)
            self._log_state("DISCARD", discarded_tile.to_mspz())
            
            self.next_turn()
        self.round_number += 1
    
    def play_game(self):
        """Loop representing a full game"""
        while not self.is_game_over():
            self.setup_round()
            self.play_round()
            self._log_state("ROUND OVER")
        self._log_state("GAME OVER")

    def _log_state(self, action_type: str, tile_id: str = None):
        """Captures the current state of the table for post-game review."""
        snapshot = {
            "round_number": self.round_number,
            "turn_index": self.turn_index,
            "tiles_remaining": self.tiles_remaining,
            "dora": copy.deepcopy(self.dora_indicators),
            "action": action_type,
            "action_tile": tile_id,
            "players": [
                {
                    "name": p.name,
                    "score": p.score,
                    "hand": p.hand.to_ids() if p.hand else [],
                    "discards": list(p.discards),
                    "is_dealer": p.is_dealer,
                    "wind": p.wind
                } for p in self.players
            ]
        }
        self.log.append(snapshot)

In [5]:
# VISUALIZE SIMULATION

class MahjongReplay:
    def __init__(self, history):
        self.history = history
        self.current_idx = 0
        self.output = widgets.Output()
        # Find all indices where a new round starts
        self.round_start_indices = self._get_round_indices()

    def _get_round_indices(self):
        indices = [i for i,n in enumerate(self.history) if n["action"] == "SETUP ROUND"]
        return indices

    def render(self):
        """Render function using direct file paths for tile assets."""
        if not self.history:
            with self.output:
                print("Error: History is empty.")
            return

        state = self.history[self.current_idx]
        with self.output:
            clear_output(wait=True)
            
            # Header Info
            field_wind = MahjongTable.BA[state['round_number']]
            
            html = f"""
            <div style="background-color: #1e1e1e; color: #d4d4d4; padding: 20px; border-radius: 10px; font-family: 'Consolas', monospace; border: 1px solid #333; width: fit-content; min-width: 650px;">
                <div style="border-bottom: 2px solid #4ec9b0; padding-bottom: 8px; margin-bottom: 15px; display: flex; justify-content: space-between; align-items: center;">
                    <span style="font-size: 1.2em; font-weight: bold; color: #4ec9b0;">
                        {field_wind} {state['round_number'] + 1} Kyoku
                    </span>
                    <span style="color: #858585;">Tiles: {state['tiles_remaining']} | Step: {self.current_idx}</span>
                </div>
            """
            
            for i, p in enumerate(state['players']):
                is_active = (i == state['turn_index'])
                active_css = "background: #252526; border: 1px solid #569cd6;" if is_active else "border: 1px solid #333;"
                
                # 1. Get the 2D list of paths from your helper
                hand_str = MSPZD(MahjongConverter.to_str(p['hand'])).notation
                path_groups = mspzd_to_file_path(hand_str)
                
                # Flatten the list: [[m...], [p...]] -> [path1, path2...]
                all_paths = [path for group in path_groups for path in group]
                # kekw=[n for n in hand_str if n.isdigit()]
                # for i,n in enumerate(kekw):
                #     print(n, all_paths[i])
                
                # 2. Build Tile HTML using direct src paths
                tile_html = ""
                for idx, path in enumerate(all_paths):
                    # Separation for the 14th (drawn) tile
                    margin = "margin-left: 14px;" if (is_active and idx == len(all_paths) - 1 and len(all_paths) == 14) else "margin-left: 1px;"
                    # Note: We use the local path directly in src
                    tile_html += f"""
                    <img src="{path}" style="height: 60px; width: 38px; {margin} border-radius: 4px; border: 1px solid #000; vertical-align: bottom;">
                    """

                seat_wind = MahjongTable.BA[p['wind']]

                html += f"""
                <div style="padding: 12px; margin: 10px 0; border-radius: 6px; {active_css}">
                    <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                        <span>
                            <b style="color: #9cdcfe; font-size: 1.1em;">[{seat_wind}]</b> {p['name']}
                            {'<span style="color: #f1c40f; font-size: 0.8em; margin-left: 8px;">▼ DEALER</span>' if p['is_dealer'] else ''}
                        </span>
                        <span style="color: #b5cea8; font-weight: bold; letter-spacing: 1px;">{p['score']:,}</span>
                    </div>
                    <div style="display: flex; align-items: center; background: #111; padding: 10px; border-radius: 4px;">
                        {tile_html if all_paths else '<span style="color:#555">Empty Hand</span>'}
                    </div>
                </div>
                """

            html += "</div>"
            display(HTML(html))

    # --- Navigation Logic ---
    def move_next_turn(self, _):
        if self.current_idx < len(self.history) - 1:
            self.current_idx += 1
            self.render()

    def move_prev_turn(self, _):
        if self.current_idx > 0:
            self.current_idx -= 1
            self.render()

    def move_next_round(self, _):
        for idx in self.round_start_indices:
            if idx > self.current_idx:
                self.current_idx = idx
                self.render()
                break

    def move_prev_round(self, _):
        for idx in reversed(self.round_start_indices):
            if idx < self.current_idx:
                self.current_idx = idx
                self.render()
                break

    def show(self):
        """Build and display the entire UI."""
        # Create buttons
        b1 = widgets.Button(description="PREV R", layout=widgets.Layout(width='80px'))
        b2 = widgets.Button(description="PREV T", layout=widgets.Layout(width='80px'))
        b3 = widgets.Button(description="NEXT T", layout=widgets.Layout(width='80px'))
        b4 = widgets.Button(description="NEXT R", layout=widgets.Layout(width='80px'))

        # Navigation logic (direct functions)
        def p_r(_): 
            for idx in reversed(self.round_start_indices):
                if idx < self.current_idx:
                    self.current_idx = idx; break
            self.render()
        
        def p_t(_): 
            self.current_idx = max(0, self.current_idx - 1); self.render()
            
        def n_t(_): 
            self.current_idx = min(len(self.history)-1, self.current_idx + 1); self.render()
            
        def n_r(_):
            for idx in self.round_start_indices:
                if idx > self.current_idx:
                    self.current_idx = idx; break
            self.render()

        b1.on_click(p_r); b2.on_click(p_t); b3.on_click(n_t); b4.on_click(n_r)

        # Trigger initial render
        self.render()
        
        # Display everything directly
        ui = widgets.VBox([widgets.HBox([b1, b2, b3, b4]), self.output])
        display(ui)

def mspzd_to_file_path(mspzd: str):
    pattern = re.compile(r"([0-9]+)([mpszdf])")
    result = {}
    matches = pattern.findall(mspzd)
    for digits, suit in matches:
        digit_list = [int(d) for d in digits]
        if suit in result:
            result[suit].extend(digit_list)
        else:
            result[suit] = digit_list
        result[suit].sort()
    # return a 2D array where each inner array contains path representing in the order of m, p, s, z, d
    ref = "mpszd"
    return [[os.path.join(TILE_IMAGE_DIR, ref[i], f"{ref[i]}{num}.png") for num in v] for i,(k,v) in enumerate(result.items())]

In [6]:
table = MahjongTable(
    starting_score=25000,
    ending_score=30000,
    num_players = 4,
    player_names = ["sakana", "yagata", "lynn", "byron"],
    game_len = 4
)
table.play_game()
game_log = table.log

replay = MahjongReplay(game_log)
replay.show()